# 02 — Resumable latest-to-Silver formatter

Load the latest Bronze export into `silver.slv_<table>`. Each source table and
export timestamp is audited in `monitoring.cfg_silver_export_load`.

- `SUCCESS` + `reload = false`: skip.
- `FAILED` or missing audit row: process on the next run.
- `reload = true`: force a successful export to run again, then reset the flag.

This means a failure on the nth table can be resumed without reloading the
tables that already completed successfully.


In [1]:
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
CFG_NOTEBOOK_NAME = "00_setup_cfg 02 03"
AUDIT_TABLE = "monitoring.cfg_silver_export_load"
LATEST_PREFIXES = ("brz_",)
STRICT_SCHEMA = True
FAIL_ON_TABLE_ERROR = True
DATE_FORMATS = ["yyyy-MM-dd", "dd/MM/yyyy", "yyyy-MM-dd'T'HH:mm:ss"]
TIME_PARSER_POLICY = "CORRECTED"
TIMESTAMP_FORMATS = [
    "yyyy-MM-dd HH:mm:ss.SSSSSS",
    "yyyy-MM-dd HH:mm:ss.SSS",
    "yyyy-MM-dd HH:mm:ss.S",
    "yyyy-MM-dd HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss.SSSSSS",
    "yyyy-MM-dd'T'HH:mm:ss.SSS",
    "yyyy-MM-dd'T'HH:mm:ss.S",
    "yyyy-MM-dd'T'HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss.SSSXXX",
    "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX",
]

# Shared configuration setup
NOTEBOOK_TIMEOUT_SECONDS = 1800


StatementMeta(, 1cd38bd0-8def-4d4e-839e-045ee7056c9a, 3, Finished, Available, Finished, False)

In [2]:
%run common_util

StatementMeta(, 1cd38bd0-8def-4d4e-839e-045ee7056c9a, 4, Finished, Available, Finished, True)

In [3]:
import re
import uuid
from collections import defaultdict
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, IntegerType, LongType, StringType, StructField, StructType, TimestampType
)
from pyspark.sql.window import Window

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.utcnow()
spark.conf.set("spark.sql.legacy.timeParserPolicy", TIME_PARSER_POLICY)


def qident(value):
    return "`" + str(value).replace("`", "``") + "`"


def normalise(value):
    return re.sub(r"[^a-z0-9]", "", (value or "").lower())


def append_rows(table_name, rows, schema):
    if rows:
        spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(table_name)


def map_data_type(pg_type):
    value = (pg_type or "").lower().strip()
    if "[]" in value:
        return "ARRAY<STRING>"
    if any(token in value for token in ("uuid", "json", "text", "character", "varchar")):
        return "STRING"
    if value in {"smallint", "int2", "integer", "int", "int4"}:
        return "INT"
    if value in {"bigint", "int8"}:
        return "BIGINT"
    match = re.search(r"(?:numeric|decimal)\s*\((\d+)\s*,\s*(\d+)\)", value)
    if match:
        precision = min(int(match.group(1)), 38)
        scale = min(int(match.group(2)), precision)
        return f"DECIMAL({precision},{scale})"
    if "numeric" in value or "decimal" in value:
        return "DECIMAL(38,18)"
    if any(token in value for token in ("double", "float", "real")):
        return "DOUBLE"
    if "boolean" in value or value == "bool":
        return "BOOLEAN"
    if value == "date":
        return "DATE"
    if "timestamp" in value:
        return "TIMESTAMP"
    return "STRING"


def first_parsed(column, formats, parser):
    return F.coalesce(*[parser(column, fmt) for fmt in formats])


def cast_column(frame, definition):
    name = definition["column_name"]
    spark_type = map_data_type(definition["data_type"])
    if name not in frame.columns:
        return F.lit(None).cast(spark_type).alias(name)
    source = F.col(qident(name))
    if spark_type == "BOOLEAN":
        clean = F.lower(F.trim(source.cast("string")))
        return (F.when(clean.isin("true", "t", "1", "yes", "y"), F.lit(True))
            .when(clean.isin("false", "f", "0", "no", "n"), F.lit(False))
            .otherwise(F.lit(None).cast("boolean")).alias(name))
    if spark_type == "DATE":
        return first_parsed(source.cast("string"), DATE_FORMATS, F.to_date).alias(name)
    if spark_type == "TIMESTAMP":
        return first_parsed(source.cast("string"), TIMESTAMP_FORMATS, F.to_timestamp).alias(name)
    if spark_type.startswith("DECIMAL") or spark_type in {"INT", "BIGINT", "DOUBLE"}:
        return F.regexp_replace(source.cast("string"), r"[^0-9eE+\.\-]", "").cast(spark_type).alias(name)
    if spark_type == "ARRAY<STRING>":
        return F.when(source.isNull(), F.lit(None).cast("array<string>"))             .otherwise(F.split(F.regexp_replace(source.cast("string"), r"^[\{\[]|[\}\]]$", ""), r"\s*,\s*")).alias(name)
    return F.trim(source.cast("string")).alias(name)


def resolve_contract(physical_table, prefixes):
    base = physical_table.lower()
    for prefix in prefixes:
        if base.startswith(prefix):
            base = base[len(prefix):]
    matches = contracts_by_table.get(normalise(base), [])
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Ambiguous table contract for {physical_table}: {matches}")
    return None


def deduplicate_frame(frame, schema_cols):
    key_columns = [c["column_name"] for c in schema_cols if (c.get("is_primary_key") or "").upper() == "YES"]
    if not key_columns:
        return frame, 0
    if "_ingestion_timestamp" in frame.columns:
        ordering = F.col("_ingestion_timestamp").cast("timestamp").desc_nulls_last()
    else:
        ordering = F.col(qident("export_date")).cast("timestamp").desc_nulls_last()
    window = Window.partitionBy(*[F.col(qident(c)) for c in key_columns]).orderBy(ordering)
    ranked = frame.withColumn("_silver_row_number", F.row_number().over(window))
    deduplicated = ranked.where(F.col("_silver_row_number") == 1).drop("_silver_row_number")
    return deduplicated, frame.count() - deduplicated.count()


def format_frame(frame, schema_cols, source_kind, source_table):
    expressions = [cast_column(frame, definition) for definition in schema_cols]
    return (frame.select(*expressions)
        .withColumn("_record_source", F.lit(source_kind))
        .withColumn("_source_table", F.lit(source_table))
        .withColumn("_silver_run_id", F.lit(RUN_ID))
        .withColumn("_silver_load_ts", F.current_timestamp()))


AUDIT_SCHEMA = StructType([
    StructField("source_kind", StringType(), False),
    StructField("source_schema", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("target_table", StringType(), True),
    StructField("export_date", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("reload", BooleanType(), False),
    StructField("attempt_count", IntegerType(), False),
    StructField("run_id", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("duplicate_key_count", LongType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("error_message", StringType(), True),
    StructField("last_updated_at", TimestampType(), True),
])


def audit_record(source_kind, source_schema, source_table, export_date):
    rows = (spark.table(AUDIT_TABLE)
        .where((F.col("source_kind") == source_kind)
            & (F.col("source_schema") == source_schema)
            & (F.col("source_table") == source_table)
            & (F.col("export_date") == F.lit(export_date).cast("timestamp")))
        .limit(1).collect())
    return rows[0].asDict() if rows else None



def target_requires_refresh(target_table, schema_cols):
    """Return True when a prior-success target is absent or contract-incomplete."""
    if not spark.catalog.tableExists(target_table):
        return True
    actual_columns = {field.name.lower() for field in spark.table(target_table).schema.fields}
    expected_columns = {definition["column_name"].lower() for definition in schema_cols}
    missing_columns = expected_columns - actual_columns
    if missing_columns:
        print(f"REFRESH {target_table}: missing contract columns {sorted(missing_columns)}")
        return True
    return False

def should_skip(source_kind, source_schema, source_table, export_date):
    record = audit_record(source_kind, source_schema, source_table, export_date)
    return bool(record and record["status"] == "SUCCESS" and not record["reload"])


def audit_begin(source_kind, source_schema, source_table, target_table, export_date):
    now = datetime.utcnow()
    existing = audit_record(source_kind, source_schema, source_table, export_date)
    attempt_count = int(existing["attempt_count"] or 0) + 1 if existing else 1
    row = [(source_kind, source_schema, source_table, target_table, export_date, "RUNNING",
            bool(existing["reload"]) if existing else False, attempt_count, RUN_ID,
            None, None, None, now, None, None, now)]
    source = spark.createDataFrame(row, AUDIT_SCHEMA)
    target = DeltaTable.forName(spark, AUDIT_TABLE)
    condition = " AND ".join([
        "t.source_kind = s.source_kind", "t.source_schema = s.source_schema",
        "t.source_table = s.source_table", "t.export_date = s.export_date",
    ])
    (target.alias("t").merge(source.alias("s"), condition)
        .whenMatchedUpdate(set={
            "target_table": "s.target_table", "status": "s.status",
            "attempt_count": "s.attempt_count", "run_id": "s.run_id",
            "started_at": "s.started_at", "ended_at": "s.ended_at",
            "error_message": "s.error_message", "last_updated_at": "s.last_updated_at",
        }).whenNotMatchedInsertAll().execute())


def audit_finish(source_kind, source_schema, source_table, target_table, export_date,
                 status, rows_read=0, rows_written=0, duplicate_count=0, error_message=None):
    now = datetime.utcnow()
    existing = audit_record(source_kind, source_schema, source_table, export_date) or {}
    row = [(source_kind, source_schema, source_table, target_table, export_date, status,
            False if status == "SUCCESS" else bool(existing.get("reload", False)),
            int(existing.get("attempt_count") or 1), RUN_ID, int(rows_read), int(rows_written),
            int(duplicate_count), existing.get("started_at") or now, now,
            error_message[:4000] if error_message else None, now)]
    source = spark.createDataFrame(row, AUDIT_SCHEMA)
    target = DeltaTable.forName(spark, AUDIT_TABLE)
    condition = " AND ".join([
        "t.source_kind = s.source_kind", "t.source_schema = s.source_schema",
        "t.source_table = s.source_table", "t.export_date = s.export_date",
    ])
    (target.alias("t").merge(source.alias("s"), condition)
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


StatementMeta(, 1cd38bd0-8def-4d4e-839e-045ee7056c9a, 5, Finished, Available, Finished, False)

In [4]:

from notebookutils import mssparkutils

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qident(SILVER_SCHEMA)}")
cfg_result = mssparkutils.notebook.run(
                CFG_NOTEBOOK_NAME,
                NOTEBOOK_TIMEOUT_SECONDS,
                {"AUDIT_TABLE": AUDIT_TABLE, "TIME_PARSER_POLICY": TIME_PARSER_POLICY}
            )
print(f"Configuration setup completed: {cfg_result}")


StatementMeta(, 1cd38bd0-8def-4d4e-839e-045ee7056c9a, 6, Finished, Available, Finished, False)

Configuration setup completed: 


In [5]:
append_rows(
    "monitoring.cfg_pipeline_run",
    [(RUN_ID, "02_silver_formatter", "SILVER", "LATEST", STARTED_AT, None, "RUNNING", 0, 0, 0, 0, None)],
    "run_id string,pipeline_name string,layer string,source_kind string,started_at timestamp,ended_at timestamp,status string,tables_succeeded int,tables_failed int,rows_read long,rows_written long,error_message string",
)


StatementMeta(, 1cd38bd0-8def-4d4e-839e-045ee7056c9a, 7, Finished, Available, Finished, False)

In [6]:
schema_df = spark.table("monitoring.cfg_schema_contract_column")
if schema_df.rdd.isEmpty():
    raise ValueError("monitoring.cfg_schema_contract_column is empty; run setup CSV bootstrap first")
schema_rows = [row.asDict(recursive=True) for row in schema_df.collect()]

contracts = defaultdict(list)
for row in schema_rows:
    if row.get("table_name") and row.get("column_name"):
        row["ordinal_position"] = int(row.get("ordinal_position") or 999999)
        contracts[row["table_name"].lower()].append(row)
for key in contracts:
    contracts[key].sort(key=lambda item: item["ordinal_position"])

contracts_by_table = defaultdict(list)
for key in contracts:
    contracts_by_table[normalise(key)].append(key)

print(f"Loaded {len(schema_rows):,} column definitions for {len(contracts):,} tables")


StatementMeta(, 1cd38bd0-8def-4d4e-839e-045ee7056c9a, 8, Finished, Available, Finished, False)

Loaded 636 column definitions for 57 tables


In [7]:
source_kind = "LATEST"
table_rows = spark.sql(f"SHOW TABLES IN {qident(BRONZE_SCHEMA)}").collect()
all_physical_tables = sorted(row.tableName for row in table_rows if not row.isTemporary)
excluded_physical_tables = excluded_etl_tables(all_physical_tables)
physical_tables = [name for name in all_physical_tables if not is_etl_excluded_table(name)]
if excluded_physical_tables:
    print(f"Excluded internal/reference Bronze tables: {excluded_physical_tables}")
metric_schema = "run_id string,layer string,source_kind string,source_object string,target_object string,rows_read long,rows_written long,duplicate_key_count long,null_primary_key_count long,recorded_at timestamp"
drift_schema = "run_id string,source_kind string,source_table string,target_table string,drift_type string,column_name string,expected_type string,actual_type string,referenced_table string,referenced_column string,drift_key string,status string,occurrence_count long,first_detected_at timestamp,last_detected_at timestamp,resolved_at timestamp,detected_at timestamp"

def drift_event_row(source_kind, source_table, target_table, drift_type,
                    column_name=None, expected_type=None, actual_type=None,
                    referenced_table=None, referenced_column=None):
    now = datetime.utcnow()
    identity = "||".join(str(value or "") for value in (
        source_kind, source_table, column_name, drift_type,
        referenced_table, referenced_column,
    ))
    drift_key = str(uuid.uuid5(uuid.NAMESPACE_URL, identity))
    return (
        RUN_ID, source_kind, source_table, target_table, drift_type,
        column_name, expected_type, actual_type, referenced_table,
        referenced_column, drift_key, "ACTIVE", 1, now, now, None, now,
    )

ok = failed = skipped = total_read = total_written = 0
errors = []

# Order tables so that FK parents are formatted before their children. The
# schema contract carries referenced_table for each FK; build a dependency
# graph over the contracted tables and topologically sort it. Alphabetical
# order alone can format a child (e.g. offer.additional_fee) before its parent
# (offer.offer), breaking downstream referential-integrity checks.
def order_tables_by_dependency(physical_tables):
    # Resolve each table's contract (metadata only, no data read) so we can
    # map normalised contract table name -> physical table.
    contract_name_to_physical = {}
    table_contract = {}
    for physical_table in physical_tables:
        contract_key = resolve_contract(physical_table, LATEST_PREFIXES)
        if contract_key is None:
            continue  # no contract: handled (and skipped) in the main loop
        table_contract[physical_table] = contract_key
        contract_name_to_physical.setdefault(
            normalise(contract_key), physical_table
        )

    # dependencies[physical] = set of physical tables that must load first.
    dependencies = {t: set() for t in physical_tables}
    for physical_table, contract_key in table_contract.items():
        for col in contracts[contract_key]:
            ref_table = (col.get("referenced_table") or "").strip()
            if not ref_table:
                continue
            parent = contract_name_to_physical.get(normalise(ref_table))
            if parent and parent != physical_table:
                dependencies[physical_table].add(parent)

    # Kahn's algorithm with deterministic (alphabetical) tie-breaking.
    ordered = []
    remaining = {t: set(deps) for t, deps in dependencies.items()}
    while remaining:
        ready = sorted(t for t, deps in remaining.items() if not deps)
        if not ready:  # cycle: break deterministically to avoid an infinite loop
            ready = [sorted(remaining)[0]]
        for physical_table in ready:
            ordered.append(physical_table)
            del remaining[physical_table]
            for deps in remaining.values():
                deps.discard(physical_table)
    return ordered


physical_tables = order_tables_by_dependency(physical_tables)
print("Dependency-ordered format: " + ", ".join(physical_tables))

for physical_table in physical_tables:
    source_table = f"{BRONZE_SCHEMA}.{physical_table}"
    target_table = None
    contract_key = None
    export_date = None
    try:
        # Read the export timestamp before contract resolution so a missing
        # table contract can still be recorded against a real source batch.
        frame = spark.table(source_table)
        if "export_date" not in frame.columns:
            raise ValueError(f"{source_table} has no export_date; rerun 01_bronze_get_latest first")

        export_date = (frame.select(F.max(F.to_timestamp("export_date")).alias("export_date"))
            .first()["export_date"])
        if export_date is None:
            raise ValueError(f"{source_table} has no valid export_date values")

        contract_key = resolve_contract(physical_table, LATEST_PREFIXES)
        if contract_key is None:
            message = f"No schema contract for {source_table}; table skipped"
            audit_finish(
                source_kind, BRONZE_SCHEMA, source_table, None, export_date,
                "SKIPPED_NO_CONTRACT", error_message=message,
            )
            append_rows(
                "monitoring.cfg_schema_drift_event",
                [drift_event_row(source_kind, source_table, None, "MISSING_TABLE_CONTRACT")],
                drift_schema,
            )
            skipped += 1
            print(f"SKIP {source_table} @ {export_date}: missing schema contract")
            continue

        contract_table = contract_key
        schema_cols = contracts[contract_key]
        target_table = f"{SILVER_SCHEMA}.slv_{contract_table}"

        if should_skip(source_kind, BRONZE_SCHEMA, source_table, export_date) and not target_requires_refresh(target_table, schema_cols):
            skipped += 1
            print(f"SKIP {source_table} @ {export_date}: already successful")
            continue

        audit_begin(source_kind, BRONZE_SCHEMA, source_table, target_table, export_date)
        batch = frame.where(F.to_timestamp("export_date") == F.lit(export_date).cast("timestamp"))
        source_count = batch.count()
        contract_columns = {c["column_name"] for c in schema_cols}
        technical_columns = {c for c in batch.columns if c.startswith("_")}
        missing = sorted(contract_columns - set(batch.columns))
        extra = sorted(set(batch.columns) - contract_columns - technical_columns)
        drift_rows = [drift_event_row(
            source_kind, source_table, target_table, "MISSING", name,
            map_data_type(next(c["data_type"] for c in schema_cols if c["column_name"] == name)),
        ) for name in missing]
        drift_rows += [drift_event_row(
            source_kind, source_table, target_table, "EXTRA", name, actual_type=dict(batch.dtypes).get(name),
        ) for name in extra]
        append_rows("monitoring.cfg_schema_drift_event", drift_rows, drift_schema)

        deduplicated, duplicate_count = deduplicate_frame(batch, schema_cols)
        formatted = format_frame(deduplicated, schema_cols, source_kind, source_table)
        formatted.write.format("delta").mode("overwrite").option("overwriteSchema", "true")             .saveAsTable(target_table)
        written = formatted.count()
        audit_finish(source_kind, BRONZE_SCHEMA, source_table, target_table, export_date,
            "SUCCESS", source_count, written, duplicate_count)
        append_rows("monitoring.cfg_table_load_metric", [(RUN_ID, "SILVER", source_kind,
            source_table, target_table, source_count, written, duplicate_count, None,
            datetime.utcnow())], metric_schema)
        ok += 1; total_read += source_count; total_written += written
        print(f"OK {source_table} -> {target_table} @ {export_date}: {written:,} rows")
    except Exception as exc:
        message = str(exc)[:4000]
        errors.append(f"{source_table}: {message}")
        failed += 1
        if export_date is not None:
            audit_finish(source_kind, BRONZE_SCHEMA, source_table, target_table, export_date,
                "FAILED", error_message=message)
        print(f"FAILED {source_table}: {message}")

status = "FAILED" if errors else "SUCCESS"
error_text = " | ".join(errors)[:4000] if errors else None
error_sql = "NULL" if error_text is None else "'" + error_text.replace("'", "''") + "'"
spark.sql(f"""UPDATE monitoring.cfg_pipeline_run SET ended_at=current_timestamp(), status='{status}',
tables_succeeded={ok}, tables_failed={failed}, rows_read={total_read}, rows_written={total_written},
error_message={error_sql} WHERE run_id='{RUN_ID}'""")
print(f"Latest Silver run {RUN_ID}: {status}; loaded={ok}, skipped={skipped}, failed={failed}")
if errors and FAIL_ON_TABLE_ERROR:
    raise RuntimeError(f"Silver formatting failed for {failed} table(s): {error_text}")


StatementMeta(, 1cd38bd0-8def-4d4e-839e-045ee7056c9a, 9, Finished, Available, Finished, False)

Excluded internal/reference Bronze tables: ['ref_KPI_Definition', 'ref_KPI_RID_linkage', 'ref_RID', 'ref_Table_Lineage']
Dependency-ordered format: framework, framework_category, holding_company, mlv_additional_fee, provider_sic_codes, provider_submission_docs, referral, referral_person_support_needs, provider, referral_category, referral_person, referral_spot_category, provider_framework, provider_home, referral_provider, offer, provider_education_provision, provider_home_age, provider_home_category, provider_home_gender, provider_home_spot_category, referral_provider_cancel_reason, referral_provider_decline_reason, referral_provider_message, additional_fee, foster_carer, foster_home, foster_transport, ipa, offer_updates, offer_view_history, supervising_social_worker, ipa_additional_fee, ipa_child, ipa_child_support_needs
SKIP bronze.framework @ 2026-08-16 13:50:50.615883: missing schema contract
SKIP bronze.framework_category @ 2026-08-15 00:00:00: already successful
SKIP bronze.hold